<img src='https://repository-images.githubusercontent.com/121802384/c355bb80-7d42-11e9-9e0e-4729609f9fbc' alt='WRF-Hydro Logo' width="15%"/>

# Lesson 4 - Run-time options for NWM configuration

## Overview
In [Lesson 2](Lesson-2-run.ipynb) we covered the basics of constructing and running a WRF-Hydro simulation using a prepared domain for the 'NWM' configuration. And in [Lesson 3](Lesson-3-visualize.ipynb), we covered how to do some basic plotting for the 2D input/output files and also plot some time series. 

In this lesson, we experiment with few different run-time options and do some basic visualization of the outputs using the Python library *xarray*.  

**NOTE: If you have not completed Lessons 2 and 3, please stop and do so now.**

### Compiling WRF-Hydro
We do not need to modify the code binary to conduct physics experiments, so we will use the previously compiled code.  

If you need to recompile the code, refer to the instructions in [Lesson 2](Lesson-2-run.ipynb).

## Experiment Setup and Baseline Simulation
In this section we will run a few different simulations with different run-time options and explore their effects on the model output.

### Step 1: Create a new template directory
We will need to create a new simulation directory for each experiment. To make this easier, we will first create a new simulation directory and use this as a template for creating multiple experiment directories.

In [ ]:
%%bash
# Make a new directory for our baseline simulation
mkdir -p ~/wrf_hydro_training/output/lesson4/run_nwm_template

# Copy our model files to the simulation directory
cp ~/wrf_hydro_training/wrf_hydro_nwm_public/build/Run/*.TBL \
  ~/wrf_hydro_training/output/lesson4/run_nwm_template
cp ~/wrf_hydro_training/wrf_hydro_nwm_public/build/Run/wrf_hydro \
  ~/wrf_hydro_training/output/lesson4/run_nwm_template

# Create symbolic links to files we will not edit
ln -sf ~/wrf_hydro_training/example_case/FORCING \
  ~/wrf_hydro_training/output/lesson4/run_nwm_template
ln -sf ~/wrf_hydro_training/example_case/NWM/DOMAIN \
  ~/wrf_hydro_training/output/lesson4/run_nwm_template
ln -sf ~/wrf_hydro_training/example_case/NWM/RESTART \
  ~/wrf_hydro_training/output/lesson4/run_nwm_template

# Copy namelist files
# We will use the DefaultParams namelists for the longer spring simulation period
cp ~/wrf_hydro_training/example_case/DefaultParams/namelist.hrldas \
  ~/wrf_hydro_training/output/lesson4/run_nwm_template
cp ~/wrf_hydro_training/example_case/DefaultParams/hydro.namelist \
  ~/wrf_hydro_training/output/lesson4/run_nwm_template

View the contents

In [ ]:
%%bash
ls ~/wrf_hydro_training/output/lesson4/run_nwm_template

We now have our template simulation directory ready to go. In each subsequent experiment we will copy this directory and edit the two namelist files *namelist.hrldas* and *hydro.namelist* to change run-time options.

### Step 2: Setup and run a baseline experiment

We now make a new run directory to run the baseline simulation with standard NWM configuration settings.

In [ ]:
%%bash
cp -r ~/wrf_hydro_training/output/lesson4/run_nwm_template \
  ~/wrf_hydro_training/output/lesson4/run_nwm_baseline

Launch the baseline run and make sure it completes successfully.

In [ ]:
%%bash
cd ~/wrf_hydro_training/output/lesson4/run_nwm_baseline
mpirun -np 2 ./wrf_hydro &> run.log

In [ ]:
%%bash
# Check that it finished successfully
tail -1 ~/wrf_hydro_training/output/lesson4/run_nwm_baseline/diag_hydro.00000

## Experiment 1 - Warm vs cold start
### Background
A model **cold start** is when you start your simulation with default initial conditions. In this case, model states are initialized from the `wrfinput.nc` file (land surface states) and `GWBUCKPARM.nc` (groundwater bucket states, if active). Often these initial values are guesses at global, physically plausible starting values. However, model states like soil moisture, groundwater levels, and stream levels need time to adjust to local climate and environment. So in most cases, we first **spinup** the model to reach an equilibrium state before launching a model experiment and evaluating output. Model states at the end of this **spinup** period can be used to **warm start** a simulation. A **warm start** is when the model simulation begins with the simulated values from a previous run. How long you need to **spinup** your model varies based on the local climate (e.g., drier climates generally take longer than wetter climates), the variables you are interested in (e.g., a fast stabilizing state like shallow soil moisture vs. a slower evolving state like deep groundwater), and local topography and soil types (e.g., sands have shorter memory than clays). 

### Objective
The goal of this experiment is to demonstrate the effect of starting a simulation from a *cold stat* vs. a *warm start*. The NWM baseline run was started from a *warm start*, so all we need to do is construct a simulation directory and edit the namelists to start from a cold start.

### Create experiment simulation directory
We will copy our template directory `~/wrf_hydro_training/output/lesson4/run_nwm_template` and edit the `namelist.hrldas` and `hydro.namelist` files to start the model from a cold start. 

In [ ]:
%%bash
cp -r ~/wrf_hydro_training/output/lesson4/run_nwm_template ~/wrf_hydro_training/output/lesson4/run_cold_start

### Edit the namelists and run the simulation
If restart files are not provided in the `namelist.hrldas` and `hydro.namelist` files then the simulation will start from a cold start. 

**Step 1: Edit the *namelist.hrldas* file**

Comment out this line `RESTART_FILENAME_REQUESTED = "RESTART/RESTART.2023030100_DOMAIN1"` in the [namelist.hrldas](../output/lesson4/run_cold_start/namelist.hrldas) by adding `!` to the start of the line. 

```
!RESTART_FILENAME_REQUESTED = "RESTART/RESTART.2023030100_DOMAIN1"
```

**Step 2: Edit the *hydro.namelist* file**

Comment out this line `RESTART_FILE  = 'RESTART/HYDRO_RST.2023-03-01_00_00_DOMAIN1'` in the [hydro.namelist](../output/lesson4/run_cold_start/hydro.namelist) by adding `!` to the start of the line. 

```
! Specify the name of the restart file if starting from restart...comment out with '!' if not...
!RESTART_FILE  = 'RESTART/HYDRO_RST.2023-03-01_00_00_DOMAIN1'
```
Tell the code to use GWBUCKPARM.nc initialization values vs. the restart file starting states by editing the `GW_RESTART` option in the *hydro.namelist*. 

```
! Specify baseflow/bucket model initialization...(0=cold start from table, 1=restart file)
GW_RESTART = 0
```

**Step 3: Run the simulation**

In [ ]:
%%bash
cd ~/wrf_hydro_training/output/lesson4/run_cold_start
mpirun -np 2 ./wrf_hydro &> run.log

**Step 4: Check that the simulation finished successfully**

In [ ]:
%%bash
tail -1 ~/wrf_hydro_training/output/lesson4/run_cold_start/diag_hydro.00000

### Results
We will now look at the differences in streamflow between our warm start simulation represented by the *run_nwm_baseline* simulation and our cold start simulation that we ran during this experiment.

We will use Python and the `xarray` library to create hydrographs and 2d plots of soil moisture. For an intro to these tools, please see [Lesson 3](Lesson-3-visualize.ipynb).

**Load the xarray python package**

In [ ]:
# Load required packages
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import datetime as dt

xr.set_options(display_style="html")
from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()

#### Hydrographs
**Load the CHANOBS streamflow datasets**

We are going to use the *CHANOBS* output files to extract the streamflow time series. *CHANOBS* output files contain information for only the grid cells where we have specified a gage is located.

In [ ]:
chanobs_baseline = xr.open_mfdataset('~/wrf_hydro_training/output/lesson4/run_nwm_baseline/*CHANOBS*',
                            combine='by_coords')
chanobs_cold = xr.open_mfdataset('~/wrf_hydro_training/output/lesson4/run_cold_start/*CHANOBS*',
                            combine='by_coords')

Read in the observations.

In [ ]:
obs = pd.read_csv('~/wrf_hydro_training/example_case/USGS_obs_2019_2023.csv',dtype=str)
obs['dateTime'] = pd.to_datetime(obs['dateTime'])
obs['streamflow_cms'] = pd.to_numeric(obs['streamflow_cms'])
obs = obs.sort_values(by='dateTime')

**Plot the hydrographs**

**Table 1. USGS Stream Gage IDs and associated WRF-Hydro feature_id indices.**

|USGS Gage ID|WRF-Hydro gage feature_id|Gage Information|
|------|:-:|---|
| 01362200 | 6189624 |https://waterdata.usgs.gov/pa/nwis/uv/?site_no=01362200|
| 01362500 | 6189754 |https://waterdata.usgs.gov/nwis/nwismap/?site_no=01362500|

In [ ]:
# Specify our evaluation gage and associated reach ID
gageid = "01362200"
fid = 6189624

In [ ]:
fig, axes = plt.subplots(ncols=1,figsize=(12, 6))
plt.suptitle('Hydrographs for warm and cold starts',fontsize=24)
chanobs_baseline.sel(feature_id = fid).streamflow.plot(label='Warm start',
                                                     color='red',
                                                     linestyle='-')
chanobs_cold.sel(feature_id = fid).streamflow.plot(label='Cold start',
                                                        color='blue',
                                                        linestyle='--')
obs[obs['site_no'] == gageid].plot(x='dateTime',
                                       y='streamflow_cms',
                                       ax=axes,
                                       label='Observed',
                                       color='gray')

plot_times = chanobs_baseline.time.data
plot_time_buffer = pd.to_timedelta(2, unit='D') 
plt.xlim([plot_times.min()-plot_time_buffer, plot_times.max()+plot_time_buffer])
plt.ylim(0,80)
plt.legend()
plt.show()

#### Soil moisture at start of simulation
The option `t0OutputFlag` in the `hydro.namelist` can be used to output models states at the start of the simulation (our warm start conditions from the RESTART and HYDRO_RST files).

```
! Option to write output files at time 0 (restart cold start time): 0=no, 1=yes (default)
t0OutputFlag = 1
```
We had this option set to 1 for both of our simulations so that we have outputs at the first timestep. 

**Load the LDASOUT datasets from the land surface model**

In [ ]:
ldasout_baseline = xr.open_mfdataset('~/wrf_hydro_training/output/lesson4/run_nwm_baseline/*LDASOUT*',
                            combine='by_coords')
ldasout_cold = xr.open_mfdataset('~/wrf_hydro_training/output/lesson4/run_cold_start/*LDASOUT*',
                            combine='by_coords')

**Plot soil moisture at start of simulation**

In [ ]:
# Select data for t=0 at surface layer
ldasout_baseline_t0 = ldasout_baseline.isel(time = 0).sel(soil_layers_stag = 0)
ldasout_cold_t0 = ldasout_cold.isel(time = 0).sel(soil_layers_stag = 0)

fig, axes = plt.subplots(ncols=2,figsize=(15, 6))
plt.subplots_adjust(wspace=.5)
plt.suptitle('Surface soil moisture states at t=0 for warm and cold starts',fontsize=24)
ldasout_baseline_t0.SOIL_M.plot(ax=axes[0],vmin=0.1,vmax=0.5)
axes[0].set_title('Warm start')
ldasout_cold_t0.SOIL_M.plot(ax=axes[1],vmin=0.1,vmax=0.5)
axes[1].set_title('Cold start')
plt.show()

In [ ]:
**Plot domain-average soil moisture time series**

In [ ]:
# Calculate the mean bottom-layer soil moisture across the domain
smois_baseline_avg = ldasout_baseline.SOIL_M.sel(soil_layers_stag = 3).mean(dim=('y','x'))
smois_cold_avg = ldasout_cold.SOIL_M.sel(soil_layers_stag = 3).mean(dim=('y','x')) 

# Plot the soil moisture time series
fig, axes = plt.subplots(ncols=1,figsize=(12, 6))
plt.suptitle('Average Soil Moisture: Bottom Layer',fontsize=24)
smois_baseline_avg.plot(label='Warm start', color='red', linestyle='-')
smois_cold_avg.plot(label='Cold start', color='blue', linestyle='--')
plt.legend()
plt.show()

### Discussion

Note that the streamflow for the cold start at the start of the simulation is 0 while the streamflow from the warm start (baseline) is > 0. Additionally, the cold start simulation begins with relatively low soil moisture (coming from the *SMOIS* variable in the `wrfinput_d01.nc` file) while the warm start simulation begins with higher soil moisture content (coming from the restart file). As a result they have different responses to the rainfall event. Since in the cold-start simulation soil is relatively drier, it has a higher capacity to store water and therefore more precipitation is partitioned to soil moisture (vs. runoff) compared to the warm-start simulation. 

This is a good example of why you almost always want to spinup your model prior to analyzing output.

## Experiment 2 - Physics Options

### Background
Noah-MP and WRF-Hydro both include multiple physics/parameterization options for different processes. You can review the Noah-MP physics options here: https://wrf-hydro.readthedocs.io/en/latest/appendices.html#a5-noah-mp-namelist-hrldas-file-with-description-of-options

#### Noah-MP physics options

DYNAMIC_VEG_OPTION: options for dynamic vegetation
- 1 -> off (use table LAI; use FVEG = SHDFAC from input)
- 2 -> on  (together with OPT_CRS = 1)
- 3 -> off (use table LAI; calculate FVEG)
- **4 -> off (use table LAI; use maximum vegetation fraction)
- **5 -> on  (use maximum vegetation fraction)
- 6 -> on  (use FVEG = SHDFAC from input)
- 7 -> off (use input LAI; use FVEG = SHDFAC from input)
- 8 -> off (use input LAI; calculate FVEG)
- 9 -> off (use input LAI; use maximum vegetation fraction)

CANOPY_STOMATAL_RESISTANCE_OPTION: options for canopy stomatal resistance
- **1 -> Ball-Berry
- 2 -> Jarvis

BTR_OPTION: options for soil moisture factor for stomatal resistance
- **1 -> Noah (soil moisture)
- 2 -> CLM  (matric potential)
- 3 -> SSiB (matric potential)

RUNOFF_OPTION: options for runoff and groundwater
- 1 -> TOPMODEL with groundwater (Niu et al. 2007 JGR)
- 2 -> TOPMODEL with an equilibrium water table (Niu et al. 2005 JGR)
- **3 -> original surface and subsurface runoff (free drainage)
- 4 -> BATS surface and subsurface runoff (free drainage)
- 5 -> Miguez-Macho&Fan groundwater scheme (Miguez-Macho et al. 2007 JGR; Fan et al. 2007 JGR) [NOT YET SUPPORTED WITH WRF-HYDRO]
- 7 -> Xinanjiang runoff scheme

SURFACE_DRAG_OPTION: options for surface layer drag coeff (CH & CM)
- **1 -> M-O
- 2 -> original Noah (Chen97)

FROZEN_SOIL_OPTION: options for frozen soil permeability
- **1 -> linear effects, more permeable (Niu and Yang, 2006, JHM)
- 2 -> nonlinear effects, less permeable (old)

SUPERCOOLED_WATER_OPTION: options for supercooled liquid water (or ice fraction)
- **1 -> no iteration (Niu and Yang, 2006 JHM)
- 2 -> Koren's iteration

RADIATIVE_TRANSFER_OPTION: options for radiation transfer
- 1 -> modified two-stream (gap = F(solar angle, 3D structure ...)<1-FVEG)
- 2 -> two-stream applied to grid-cell (gap = 0)
- **3 -> two-stream applied to vegetated fraction (gap=1-FVEG)

SNOW_ALBEDO_OPTION: options for ground snow surface albedo
- **1 -> BATS
- 2 -> CLASS

PCP_PARTITION_OPTION: options for partitioning  precipitation into rainfall & snowfall
- **1 -> Jordan (1991)
- 2 -> BATS: when SFCTMP<TFRZ+2.2
- 3 -> SFCTMP < TFRZ
- 4 -> Use WRF microphysics output

TBOT_OPTION: options for lower boundary condition of soil temperature
- 1 -> zero heat flux from bottom (ZBOT and TBOT not used)
- **2 -> TBOT at ZBOT (8m) read from a file (original Noah)

TEMP_TIME_SCHEME_OPTION: options for snow/soil temperature time scheme (only layer 1)
- 1 -> semi-implicit; flux top boundary condition
- 2 -> full implicit (original Noah); temperature top boundary condition
- **3 -> same as 1, but FSNO for TS calculation (generally improves snow; v3.7)

GLACIER_OPTION: options for glacier treatment
- 1 -> include phase change of ice
- **2 -> ice treatment more like original Noah (slab)

SURFACE_RESISTANCE_OPTION: options for surface resistent to evaporation/sublimation
- 1 -> Sakaguchi and Zeng, 2009
- 2 -> Sellers (1992)
- 3 -> adjusted Sellers to decrease RSURF for wet soil
- **4 -> option 1 for non-snow; rsurf = rsurf_snow for snow (set in MPTABLE); AD v3.8

IMPERV_OPTION: options for impervious adjustment for surface runoff partitioning
- 0 -> none
- 1 -> adjust based on impervious fraction
- 2 -> adjust based on effective impervious fraction from Alley & Veenhuis
- **9 -> original formulation (varies based on runoff option)

WRF-Hydro also includes different routing options. Some require specific geospatial inputs, so need to be decided before running the GIS Preprocessor.
You can see runtime options here: https://wrf-hydro.readthedocs.io/en/latest/appendices.html#a6-wrf-hydro-hydro-namelist-file-with-description-of-options

#### WRF-Hydro physics options

SUBRTSWCRT: Turn on/off subsurface routing module
- 1 -> subsurface routing module is active
- 0 -> subsurface routing module is off

OVRTSWCRT: Turn on/off overland routing module
- 1 -> overland routing module is active
- 0 -> overland routing module is off

rt_option: Specify overland flow routing option
- 1 -> steepest descent
- 2 -> multi-directional (CASC2D, currently not supported)

imperv_adj: Specify whether to adjust overland flow parameters based on imperviousness. When overland routing is active and an imperviousness grid is provided in Fulldom_hires.nc, you can use this option to reduce the overland roughness and maximum retention depth based on the impervious fraction.
- 0 -> no adjustment
- 1 -> activate parameter adjustments

CHANRTSWCRT: Switch to activate channel routing
- 1 -> channel routing module is active
- 0 -> channel routing module is off

channel_option: If channel routing module is active, select which option to use.
- 1 -> Muskingum reach
- 2 -> Muskingum-Cunge reach
- 3 -> Diffusive wave gridded

compound_channel: If using channel_option=2, activate the compound channel formulation? This option is currently only supported if using Muskingum-Cunge reach-based routing with UDMP=1. This option also requires additional parameters in the routelink file.
- .FALSE. -> compound channel is off
- .TRUE. -> compound channel is on

channel_loss_option: Switch to activate channel-loss option (requires Kchan in Routelink). Note that the channel loss scheme currently only works for Muskingum-Cunge reach-based channel routing. Also note that activating channel loss will create a sink in the model, so the water budget will not close.
- 0 -> channel loss is off
- 1 -> channel loss is on

lake_option: Set the lake/reservoir option. Note that different options may require different domain/parameter/input files. Option 0 (lakes off) will not generate reasonable results for gridded channel routing domains where lake cells mask out channel cells.
- 0 -> lakes off
- 1 -> level pool (typical default)
- 2 -> passthrough
- 3 -> reservoir DA

GWBASESWCRT: Set the groundwater bucket module option.
- 0 -> off (Note: Option 0 creates a sink at the bottom of the soil column (water draining from the bottom of the soil column leaves the system, so note that this option will not have water balance closure).
- 1 -> exponential bucket model
- 2 -> pass-through (bypass the bucket model and dump all flow from the bottom of the soil column directly into the channel)
- 4 -> exponential bucket model where the coefficient is scaled by catchment area (only works for reach-based routing with UDMP=1.)

UDMP_OPT: Whether to activate user-defined mapping, such NHDPlus
- 0 -> off
- 1 -> use a "user-defined mapping" scheme to translate between terrain and groundwater flow and reaches, e.g., NHDPlus.

### Create experiment simulation directory
First we will copy our template directory `~/wrf_hydro_training/output/lesson4/run_nwm_template` to create a new simulation directory for this experiment.

In [ ]:
%%bash
cp -r ~/wrf_hydro_training/output/lesson4/run_nwm_template \
~/wrf_hydro_training/output/lesson4/run_noahmp_mods

### Edit the namelist and run the simulation

**Step 1: Edit the `namelist.hrldas`**

Change the Noah-MP vegetation option from using a lookup table for LAI (option 4, where dynamic vegetation is off and instead the model uses table LAI and the maximum vegetation fraction input as specified in the wrfinput file) to using Noah-MP's dynamic vegetation model (option 2).

```
DYNAMIC_VEG_OPTION                = 2
```

**Step 2: Run the simulation**

In [ ]:
%%bash
cd ~/wrf_hydro_training/output/lesson4/run_noahmp_mods
mpirun -np 2 ./wrf_hydro &> run.log

tail -1 ~/wrf_hydro_training/output/lesson4/run_noahmp_mods/diag_hydro.00000

### Results
We will now look at the differences in streamflow between our baseline run and our experiment with alternative Noah-MP parameterizations.

#### Hydrographs
**Load the CHANOBS streamflow datasets**

Again, we use the CHANOBS files to import the streamflow timeseries at specified gage locations.

In [ ]:
chanobs_baseline = xr.open_mfdataset('~/wrf_hydro_training/output/lesson4/run_nwm_baseline/*CHANOBS*',
                            combine='by_coords')
chanobs_noahmp_mods = xr.open_mfdataset('~/wrf_hydro_training/output/lesson4/run_noahmp_mods/*CHANOBS*',
                            combine='by_coords')

**Plot the hydrographs**

In [ ]:
fig, axes = plt.subplots(ncols=1,figsize=(12, 6))
plt.suptitle('Hydrographs for NWM and Alternative Noah-MP Physics',fontsize=24)
chanobs_baseline.sel(feature_id = fid).streamflow.plot(label='NWM',
                                                     color='red',
                                                     linestyle='-')
chanobs_noahmp_mods.sel(feature_id = fid).streamflow.plot(label='Noah-MP Mods',
                                                        color='blue',
                                                        linestyle='--')
obs[obs['site_no'] == gageid].plot(x='dateTime',
                                       y='streamflow_cms',
                                       ax=axes,
                                       label='Observed',
                                       color='gray')

plot_times = chanobs_baseline.time.data
plot_time_buffer = pd.to_timedelta(2, unit='D') 
plt.xlim([plot_times.min()-plot_time_buffer, plot_times.max()+plot_time_buffer])
plt.ylim(0,80)
plt.legend()
plt.show()

#### Snow
We will explore modeled snow to see how it was impacted by the new dynamic vegetation parameterization. We first calculate a domain-wide mean SWE time series and then plot the two simulations.

In [ ]:
ldasout_baseline = xr.open_mfdataset('~/wrf_hydro_training/output/lesson4/run_nwm_baseline/*.LDASOUT*',
                            combine='by_coords')
ldasout_noahmp_mods = xr.open_mfdataset('~/wrf_hydro_training/output/lesson4/run_noahmp_mods/*.LDASOUT*',
                            combine='by_coords')

In [ ]:
# Calculate the average SWE across the domain
swe_baseline_avg = ldasout_baseline.SNEQV.mean(dim=('y','x'))
swe_mods_avg = ldasout_noahmp_mods.SNEQV.mean(dim=('y','x')) 

# Plot the SWE time series
fig, axes = plt.subplots(ncols=1,figsize=(12, 6))
plt.suptitle('SWE',fontsize=24)
swe_baseline_avg.plot(label='NWM', color='red', linestyle='-')
swe_mods_avg.plot(label='Noah-MP Mods', color='blue', linestyle='--')
plt.legend()
plt.show()

### Discussion
The alternative Noah-MP dynamic vegetation parameterization results in more snow accumulation leading to delayed melt and resulting streamflow.

# Next up - Exploring model parameters
This concludes lesson 4. 

In the [next lesson](Lesson-5-land-surface.ipynb), we will explore terrain physics options and parameters.

**IT IS BEST TO EITHER SHUTDOWN THIS LESSON OR CLOSE IT BEFORE PROCEEDING TO THE NEXT LESSON TO AVOID POSSIBLY EXCEEDING ALLOCATED MEMORY. Shutdown the lesson be either closing the browser tab for the lesson or selecting `Kernel -> Shut Down Kernel` in JupyterLab.**

© UCAR 2025